# Metabolite–Target Pair Analysis for Cancer Cell-Cell Communication

**File**: `human_database_merge_unique_metab_target_pairs_with_HMDB_Info.csv`  
**Goal**: Profile the interaction network, identify cancer-relevant hubs, characterize interaction biology, and prioritize pairs for downstream single-cell RNA-sequencing (scRNAseq) and TCGA clinical integration.

---
## 1. Setup

In [ ]:
import pandas as pd
import numpy as np
import matplotlib
import matplotlib.pyplot as plt
%matplotlib inline
import seaborn as sns
from collections import Counter
import os

sns.set_theme(style='whitegrid', palette='muted')
plt.rcParams.update({'figure.figsize': (12,6), 'font.size': 12, 'axes.titlesize': 16, 'axes.labelsize': 14})

In [ ]:
# ==========================================
# ⚙️ USER PARAMETERS (Confidence Filter & Export Options)
# ==========================================
# Choose which metabolite confidence tiers to retain for downstream analysis:
# - 'all'            -> Retain all target pairs (Tiers 1, 2, and 3)
# - 'tier_1'         -> Retain only Tier 1 (High Confidence) pairs (4+ databases)
# - 'tier_1_and_2'   -> Retain Tier 1 and Tier 2 (High & Medium Confidence) pairs
TIER_FILTER = 'all'  # DEFAULT: 'all'. Change to 'tier_1' or 'tier_1_and_2' to filter!

# Full Notebook HTML Report Export Toggle:
# - True  -> Automatically exports the entire notebook (Markdown, Code, Tables, Figures) to styled HTML
# - False -> Disables automatic HTML export
SAVE_AS_HTML = True  # DEFAULT: False. Change to True to export the entire notebook!

## 2. Load Data

### Purpose
To ingest the consolidated metabolite-target interaction pair database, which has been simplified and enriched with HMDB annotation metadata. Loading this data enables structural and functional network analysis across all integrated databases (Celllinker2, MRCLinkDB, MEBOCOST, and CellPhoneDBv5).

### How to Interpret the Output
- **Interaction Pairs count:** Indicates the size and complexity of the metabolic interactome.
- **Unique Metabolites & Targets count:** Reveals the diversity of the metabolic signals (ligands) and their corresponding molecular receptors/sensors in the TME.
- **DataFrame Preview:** Review the column headers to verify successful mapping of chemical identifiers (e.g., SMILES, AVERAGE_MASS), provenance databases, and clinical metadata.

In [ ]:
df = pd.read_csv('../output/human_database_merge_unique_metab_target_pairs_with_HMDB_Info.csv', low_memory=False)
print(f'Loaded {len(df):,} interaction pairs | {df["HMDB_ID"].nunique()} unique HMDB metabolites ({df["Metabolite_Name"].nunique()} names) | {df["Target"].nunique()} targets | {df.shape[1]} columns')

# ==========================================
# ⚙️ CONFIDENCE TIER MAPPING & FILTERING
# ==========================================
print("\nAssigning and mapping confidence tiers...")

# 1. Assign Pair Confidence Tier
def assign_tier(row):
    score = row['databases_count']
    has_smiles = pd.notna(row['SMILES'])
    has_mass = pd.notna(row['AVERAGE_MASS'])
    if score >= 4 and has_smiles and has_mass:
        return 'Tier 1 (High)'
    elif score >= 2 and has_smiles:
        return 'Tier 2 (Medium)'
    else:
        return 'Tier 3 (Low)'

df['Pair_Confidence_Tier'] = df.apply(assign_tier, axis=1)
df['Confidence_Tier'] = df['Pair_Confidence_Tier']  # Keep as alias for backward compatibility

# 2. Map Metabolite Confidence Tier from the unique metabolite dataset
try:
    df_metab_ref = pd.read_csv('../output/human_database_merge_unique_metab_with_HMDB_Info.csv', low_memory=False)
    df_metab_ref['Metabolite_Confidence_Tier'] = df_metab_ref.apply(assign_tier, axis=1)
    
    # Create mapping dictionaries
    hmdb_tier_map = dict(zip(
        df_metab_ref['HMDB_ID'].astype(str).str.strip(),
        df_metab_ref['Metabolite_Confidence_Tier']
    ))
    name_tier_map = dict(zip(
        df_metab_ref['Metabolite_Name'].astype(str).str.lower().str.strip(),
        df_metab_ref['Metabolite_Confidence_Tier']
    ))
    
    # Apply hybrid mapping
    df['clean_hmdb'] = df['HMDB_ID'].astype(str).str.split(',').str[0].str.strip()
    df['clean_name'] = df['Metabolite_Name'].astype(str).str.lower().str.strip()
    
    tier_by_hmdb = df['clean_hmdb'].map(hmdb_tier_map)
    tier_by_name = df['clean_name'].map(name_tier_map)
    
    df['Metabolite_Confidence_Tier'] = tier_by_hmdb.fillna(tier_by_name)
    df['Metabolite_Confidence_Tier'] = df['Metabolite_Confidence_Tier'].fillna('Tier 3 (Low)')
    
    # Drop temp columns
    df = df.drop(columns=['clean_hmdb', 'clean_name'])
except Exception as e:
    print(f"⚠️ Warning: Could not map Metabolite Confidence Tiers: {e}")
    df['Metabolite_Confidence_Tier'] = 'Tier 3 (Low)'

# Apply filter based on TIER_FILTER parameter
if TIER_FILTER == 'tier_1':
    allowed_tiers = ['Tier 1 (High)']
elif TIER_FILTER == 'tier_1_and_2':
    allowed_tiers = ['Tier 1 (High)', 'Tier 2 (Medium)']
else:
    allowed_tiers = ['Tier 1 (High)', 'Tier 2 (Medium)', 'Tier 3 (Low)']

initial_count = len(df)
df = df[df['Confidence_Tier'].isin(allowed_tiers)].copy()
filtered_count = len(df)

print(f"✅ Confidence Tier Filter Applied: Retained {allowed_tiers}")
print(f"   -> Original pairs: {initial_count:,}")
print(f"   -> Retained pairs: {filtered_count:,} ({filtered_count/initial_count*100:.1f}%)")
print(f"   -> Unique HMDB metabolites remaining: {df['HMDB_ID'].nunique():,} ({df['Metabolite_Name'].nunique():,} names)")
print(f"   -> Unique targets remaining: {df['Target'].nunique():,}\n")

# Print remaining breakdown
print("Breakdown of remaining interaction pairs (Pair Confidence):")
for tier, count in df['Pair_Confidence_Tier'].value_counts().items():
    print(f"  * {tier}: {count:,} pairs")

print("\nBreakdown of remaining interaction pairs (Metabolite Confidence):")
for tier, count in df['Metabolite_Confidence_Tier'].value_counts().items():
    print(f"  * {tier}: {count:,} pairs")

df.head()

In [ ]:
df.columns

In [ ]:
pd.unique(df["Disease"])

## 3. Network Overview

### 3.1. Degree Distribution

### Purpose
In multicellular systems, metabolite-protein interactions form a complex bipartite network. Profiling the degree distribution (how many targets interact with a single metabolite, and how many metabolites bind to a single target) is essential to characterize the structural topology of the metabolic connectome. This helps us understand whether the communication network is scale-free (dominated by a few key hubs) or uniform.

### How to Interpret the Results
- **Metabolite Degree Distribution (Left Plot):** A highly right-skewed distribution indicates that most metabolites are highly specific, interacting with only 1 or 2 protein targets (e.g., specialized paracrine cues), while a small minority act as major broad-acting hubs (interacting with 50+ targets).
- **Target Degree Distribution (Right Plot):** Skewness here represents multi-functional receptors, enzymes, or solute carriers that sense a wide variety of metabolic changes in the tumor microenvironment (TME). These constitute major metabolic sensing checkpoints.

In [ ]:
met_degree = df.groupby('Metabolite_Name')['Target'].nunique().sort_values(ascending=False)
tgt_degree = df.groupby('Target')['Metabolite_Name'].nunique().sort_values(ascending=False)

fig, axes = plt.subplots(1, 2, figsize=(16, 6))
for ax, data, title, color in [(axes[0], met_degree, 'Metabolite Degree (# unique targets)', 'steelblue'),
                                (axes[1], tgt_degree, 'Target Degree (# unique metabolites)', 'coral')]:
    ax.hist(data.values, bins=50, color=color, edgecolor='white', alpha=0.85)
    ax.set_xlabel('Degree'); ax.set_ylabel('Count'); ax.set_title(title)
    ax.axvline(data.median(), color='black', ls='--', label=f'median={data.median():.0f}')
    ax.legend()
plt.tight_layout(); plt.show()

print(f'Metabolite degree: median={met_degree.median():.0f}, max={met_degree.max()} ({met_degree.idxmax()})')
print(f'Target degree: median={tgt_degree.median():.0f}, max={tgt_degree.max()} ({tgt_degree.idxmax()})')

### 3.2. Top 20 Hub Metabolites & Targets

### Purpose
To systematically identify the most connected signaling players (hubs) in the network. In cancer systems biology, network hubs often represent critical signaling bottlenecks, metabolic sensors, or broad-spectrum therapeutic targets.

### How to Interpret the Results
- **Top Hub Metabolites (Left Plot):** These represent high-influence signaling molecules (such as Glutamate, ATP, or Adenosine) that act as "master keys" in the TME. Targeting these or their transporters can profoundly disrupt multi-cellular tumor-stromal signaling cascades.
- **Top Hub Targets (Right Plot):** These are proteins (receptors, transporters, or enzymes) that bind a large diversity of metabolites. A target appearing at the top of this list is a master metabolic sensor (such as GPCRs or broad-spectrum solute carriers) integrating multiple microenvironmental cues to regulate cellular state.

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(18, 8))
sns.barplot(x=met_degree.head(20).values, y=met_degree.head(20).index, palette='Blues_r', hue=met_degree.head(20).index, legend=False, ax=axes[0])
axes[0].set_title('Top 20 Hub Metabolites'); axes[0].set_xlabel('# Unique Targets')
sns.barplot(x=tgt_degree.head(20).values, y=tgt_degree.head(20).index, palette='Reds_r', hue=tgt_degree.head(20).index, legend=False, ax=axes[1])
axes[1].set_title('Top 20 Hub Targets'); axes[1].set_xlabel('# Unique Metabolites')
plt.tight_layout(); plt.show()

---

## 4. Database Provenance & Confidence

### 4.1. Cross-Database Validation of Pairs

### Purpose
Ligand-target databases are compiled using different curation strategies, literature extraction methods, and prediction algorithms. This analysis assesses the degree of consensus across different primary databases. High database overlap serves as a proxy for empirical confidence and biological reproducibility.

### How to Interpret the Results
- **Single-Database Support:** Pairs supported by only 1 database are more likely to contain false positives or context-specific interactions.
- **Multi-Database Consensus (2+ or 3+ databases):** Represents highly validated core interactions with high biological confidence. These are the prime candidates for high-fidelity downstream modeling or wet-lab validation.

In [ ]:
# Custom UpSet Plot for Cross-Database Validation of Pairs
import numpy as np
from matplotlib.gridspec import GridSpec

# 1. Identify all unique databases and calculate their set sizes
all_dbs = sorted(list(set(df['database'].str.split(', ').explode().dropna().unique())))
set_sizes = {db: df['database'].str.contains(db, na=False, regex=False).sum() for db in all_dbs}
sorted_dbs = sorted(all_dbs, key=lambda x: set_sizes[x], reverse=True)

# 2. Identify all intersections (database combinations) and count them
def normalize_combo(val):
    if pd.isna(val): return ()
    dbs = [d.strip() for d in val.split(',')]
    return tuple(sorted(dbs))

df_temp = df.copy()
df_temp['combo'] = df_temp['database'].apply(normalize_combo)
combo_counts = df_temp['combo'].value_counts()
combo_counts = combo_counts[combo_counts > 0]
sorted_combos = combo_counts.index.tolist()
combo_sizes = combo_counts.values.tolist()

# 3. Create the multi-panel GridSpec figure for the UpSet plot
fig = plt.figure(figsize=(14, 8))
gs = GridSpec(2, 2, height_ratios=[3, 2], width_ratios=[1.5, 4.5], wspace=0.08, hspace=0.06)

ax_top = fig.add_subplot(gs[0, 1])          # Top right: Intersection size bar plot
ax_matrix = fig.add_subplot(gs[1, 1], sharex=ax_top) # Bottom right: Connection matrix
ax_left = fig.add_subplot(gs[1, 0])         # Bottom left: Set size bar plot

# --- Top Plot: Intersection Sizes ---
bars = ax_top.bar(range(len(sorted_combos)), combo_sizes, color='#1e3a8a', width=0.55, edgecolor='#1e293b', linewidth=0.5)
ax_top.set_ylabel('Intersection Size\n(# Pairs)', fontsize=11, fontweight='semibold')
ax_top.set_title('Cross-Database Validation of Pairs', fontsize=15, fontweight='bold', pad=15)
ax_top.tick_params(axis='x', which='both', bottom=False, labelbottom=False)
ax_top.spines['top'].set_visible(False)
ax_top.spines['right'].set_visible(False)
ax_top.spines['bottom'].set_visible(False)
ax_top.grid(axis='y', linestyle='--', alpha=0.3)

for bar in bars:
    height = bar.get_height()
    ax_top.text(bar.get_x() + bar.get_width()/2., height + max(combo_sizes)*0.015, f'{int(height):,}', ha='center', va='bottom', fontsize=8.5, fontweight='semibold', color='#0f172a')

# --- Left Plot: Set Sizes ---
db_indices = {db: i for i, db in enumerate(sorted_dbs)}
left_sizes = [set_sizes[db] for db in sorted_dbs]
ax_left.barh(range(len(sorted_dbs)), left_sizes, color='#475569', height=0.55, edgecolor='#1e293b', linewidth=0.5)
ax_left.set_xlabel('Set Size (# Pairs)', fontsize=11, fontweight='semibold')
ax_left.set_yticks(range(len(sorted_dbs)))
ax_left.set_yticklabels(sorted_dbs, fontsize=11, fontweight='bold', color='#1e293b')
ax_left.invert_xaxis()
ax_left.spines['top'].set_visible(False)
ax_left.spines['left'].set_visible(False)
ax_left.spines['bottom'].set_visible(False)
ax_left.tick_params(axis='y', left=False)
ax_left.grid(axis='x', linestyle='--', alpha=0.3)

# --- Bottom Plot: Connection Matrix ---
ax_matrix.set_ylim(-0.5, len(sorted_dbs) - 0.5)
ax_matrix.set_yticks(range(len(sorted_dbs)))
ax_matrix.set_yticklabels([])
ax_matrix.spines['top'].set_visible(False)
ax_matrix.spines['right'].set_visible(False)
ax_matrix.spines['bottom'].set_visible(False)
ax_matrix.spines['left'].set_visible(False)
ax_matrix.tick_params(axis='both', which='both', left=False, bottom=False, labelbottom=False)

for i in range(len(sorted_dbs)):
    ax_matrix.axhline(i, color='#cbd5e1', linestyle='-', alpha=0.5, zorder=0)
    ax_matrix.scatter(range(len(sorted_combos)), [i]*len(sorted_combos), color='#e2e8f0', s=150, edgecolors='#cbd5e1', linewidths=0.5, zorder=1)

for col_idx, combo in enumerate(sorted_combos):
    active_indices = [db_indices[db] for db in combo if db in db_indices]
    if len(active_indices) > 0:
        if len(active_indices) > 1:
            ax_matrix.plot([col_idx, col_idx], [min(active_indices), max(active_indices)], color='#0f172a', linewidth=3, zorder=2)
        ax_matrix.scatter([col_idx]*len(active_indices), active_indices, color='#0f172a', s=180, edgecolors='#0f172a', linewidths=0.5, zorder=3)

plt.tight_layout()
plt.show()

print(f'\nHigh-confidence pairs (3+ databases): {(df["databases_count"]>=3).sum():,} ({(df["databases_count"]>=3).mean()*100:.1f}%)')


---

### 4.2. Literature Evidence, Temporal Dynamics & PubMed References Validation

### Purpose
While database consensus indicates standard curation agreement, empirical validation from peer-reviewed scientific literature is the ultimate standard of biological confidence. This section evaluates metabolite-target interaction pairs against empirical literature support, cleaning and parsing PubMed IDs (PMIDs) and their corresponding **publication years** to explore the historical depth and recency of research backing our database.

### Analytical Pillars
1. **Evidence Rate by Tier:** Verifying if our confidence tiers (Tiers 1, 2, 3) map to active literature. Higher tiers should exhibit high citation rates, validating the classification logic.
2. **Research Density (Top Hubs):** Identifying the specific metabolite-target pairs with the highest PMID citation counts to reveal the central scientific communication nodes.
3. **Temporal Dynamics (Publication Years):** Exploring the historical trajectory of research from legacy classical findings (e.g., 1980s or earlier) to cutting-edge recent discoveries (2020-2026), mapping active vs. historical interaction nodes.

In [ ]:
# ==============================================================================
# 📚 4.2. LITERATURE EVIDENCE (PMID/REFERENCES) & TEMPORAL ANALYSIS
# ==============================================================================
import re
import os
import numpy as np
import pandas as pd
%matplotlib inline
import matplotlib.pyplot as plt
import seaborn as sns

# ⚙️ CONFIGURABLE PARAMETER
TOP_N_PAIRS = 50  # Customize the number of top referenced pairs to display in Panel B

print(f"Running Literature Evidence & Temporal Analysis (Top {TOP_N_PAIRS} Pairs in Panel B)...\n")

# 1. Load PubMed Scraped Metadata for Publication Year mapping
pubmed_results_path = '../input/pubmed_results.csv'
if os.path.exists(pubmed_results_path):
    pubmed_df = pd.read_csv(pubmed_results_path, low_memory=False)
    # Standardize PMID to string for robust mapping
    pubmed_df['PMID'] = pubmed_df['PMID'].astype(str).str.strip()
    
    # Standardize Year column (extract numeric 4-digit year)
    def clean_year(y):
        if pd.isna(y):
            return np.nan
        y_str = str(y).strip()
        matches = re.findall(r'\b(19\d{2}|20\d{2})\b', y_str)
        if matches:
            return int(matches[0])
        return np.nan
        
    pubmed_df['Clean_Year'] = pubmed_df['Year'].apply(clean_year)
    pmid_to_year = dict(zip(pubmed_df['PMID'], pubmed_df['Clean_Year']))
    print(f"\u2705 Loaded {len(pubmed_df):,} unique PMIDs from PubMed metadata with publication years.")
else:
    pmid_to_year = {}
    print("\u26a0\ufe0f Warning: 'input/pubmed_results.csv' not found. Temporal analysis will be limited.")

# 2. Clean, Coalesce, and Standardize PMIDs Row-wise
def clean_row_pmids(row):
    all_pmids = set()
    # Extract PMIDs from PMID and Evidence columns
    for col in ['PMID', 'Evidence']:
        if col in row and pd.notna(row[col]):
            val_str = str(row[col]).strip()
            # Split by common separators: semicolon, comma, pipe, space, slash
            for chunk in re.split(r'[;|,\s/|]+', val_str):
                chunk = chunk.strip()
                if chunk.endswith('.0'):
                    chunk = chunk[:-2]
                if chunk.isdigit() and len(chunk) >= 5:
                    all_pmids.add(chunk)
                elif chunk and chunk.lower() not in ['nan', 'none', 'null']:
                    clean_num = re.findall(r'\b\d{5,}\b', chunk)
                    for num in clean_num:
                        all_pmids.add(num)
                        
    # Extract PMIDs from Text_Evidence (typically paragraph text with citations)
    if 'Text_Evidence' in row and pd.notna(row['Text_Evidence']):
        val_str = str(row['Text_Evidence']).strip()
        matches = re.findall(r'(?i)(?:pmid|pubmed)\s*:?\s*(\d+)', val_str)
        for m in matches:
            if len(m) >= 5:
                all_pmids.add(m)
                
    if not all_pmids:
        return np.nan
    return ';'.join(sorted(list(all_pmids), key=lambda x: int(x) if x.isdigit() else x))

# Apply the cleaning function to the active dataframe df
df['Cleaned_PMID'] = df.apply(clean_row_pmids, axis=1)
df['evidence_count'] = df['Cleaned_PMID'].apply(lambda x: len([p for p in str(x).split(';') if p.strip()]) if pd.notna(x) else 0)
df['has_evidence'] = df['evidence_count'] > 0

# 3. Map PMIDs to Publication Years for Temporal Analytics
def get_row_years(pmid_str):
    if pd.isna(pmid_str):
        return []
    years = []
    for p in str(pmid_str).split(';'):
        p = p.strip()
        if p in pmid_to_year and pd.notna(pmid_to_year[p]):
            years.append(int(pmid_to_year[p]))
    return sorted(years)

df['evidence_years'] = df['Cleaned_PMID'].apply(get_row_years)
df['earliest_year'] = df['evidence_years'].apply(lambda x: min(x) if x else np.nan)
df['latest_year'] = df['evidence_years'].apply(lambda x: max(x) if x else np.nan)
df['median_year'] = df['evidence_years'].apply(lambda x: np.nanmedian(x) if x else np.nan)

# 4. Calculate Global Temporal Metrics
all_pmids_set = set()
for p in df['Cleaned_PMID'].dropna():
    all_pmids_set.update([x.strip() for x in str(p).split(';') if x.strip()])
num_unique_pmids = len(all_pmids_set)

unique_pmid_years = []
for p in all_pmids_set:
    if p in pmid_to_year and pd.notna(pmid_to_year[p]):
        unique_pmid_years.append(int(pmid_to_year[p]))

total_pairs = len(df)
with_evidence = df['has_evidence'].sum()
pct_evidence = (with_evidence / total_pairs) * 100 if total_pairs > 0 else 0

print(f"\u2705 Literature Analysis Complete:")
print(f"   -> Total unique interaction pairs in database: {total_pairs:,}")
print(f"   -> Pairs supported by literature: {with_evidence:,} ({pct_evidence:.2f}%)")
print(f"   -> Total unique PMIDs cited: {num_unique_pmids:,}")

if unique_pmid_years:
    min_yr = min(unique_pmid_years)
    max_yr = max(unique_pmid_years)
    med_yr = int(np.nanmedian(unique_pmid_years)) if not np.isnan(np.nanmedian(unique_pmid_years)) else np.nan
    print(f"   -> Temporal span of cited literature: {min_yr} to {max_yr} (Median: {med_yr})")
    
    # Recency statistics (2020 onwards)
    recent_pmids = sum(1 for y in unique_pmid_years if y >= 2020)
    recent_pairs = sum(1 for y in df['latest_year'].dropna() if y >= 2020)
    print(f"   -> Modern research (2020+): {recent_pmids} unique PMIDs supporting {recent_pairs} pairs")
else:
    med_yr = np.nan

# 5. Create Premium Three-Panel Figure
sns.set_theme(style="whitegrid")
fig, axes = plt.subplots(1, 3, figsize=(22, 10))

# --- Panel A: Literature Evidence Rate by Confidence Tier ---
tier_stats = df.groupby('Confidence_Tier')['has_evidence'].mean() * 100
tier_stats = tier_stats.reindex(['Tier 1 (High)', 'Tier 2 (Medium)', 'Tier 3 (Low)']).fillna(0)

colors_a = ['#1a73e8', '#8ab4f8', '#ffad46']  # Harmonious custom palette
bars_a = axes[0].bar(tier_stats.index, tier_stats.values, color=colors_a, edgecolor='black', width=0.5)

# Add values on top of bars
for bar in bars_a:
    yval = bar.get_height()
    axes[0].text(bar.get_x() + bar.get_width()/2.0, yval + 1.5, f'{yval:.1f}%', ha='center', va='bottom', fontweight='bold', fontsize=11)

axes[0].set_ylim(0, 108)
axes[0].set_ylabel('Pairs with Literature Evidence (%)', fontweight='bold', fontsize=12)
axes[0].set_title('Literature Evidence Rate by Confidence Tier', fontweight='bold', fontsize=13, pad=15)
axes[0].tick_params(axis='both', which='major', labelsize=11)

# --- Panel B: Top N Pairs by Reference Count ---
top_pairs = df.sort_values(by='evidence_count', ascending=False).head(TOP_N_PAIRS).copy()
top_pairs['Pair_Label'] = top_pairs['Metabolite_Name'] + ' \u2194 ' + top_pairs['Target']

colors_b = sns.color_palette("flare", n_colors=TOP_N_PAIRS)
bars_b = axes[1].barh(top_pairs['Pair_Label'], top_pairs['evidence_count'], color=colors_b, edgecolor='black')

# Add values on right of horizontal bars
for bar in bars_b:
    xval = bar.get_width()
    axes[1].text(xval + max(1, xval*0.02), bar.get_y() + bar.get_height()/2.0, f'{int(xval):,}', ha='left', va='center', fontweight='bold', fontsize=10)

axes[1].invert_yaxis()  # top-down ranking
axes[1].set_xlabel('Number of Unique PMIDs', fontweight='bold', fontsize=12)
axes[1].set_title(f'Top {TOP_N_PAIRS} Pairs by Reference Count', fontweight='bold', fontsize=13, pad=15)
axes[1].tick_params(axis='both', which='major', labelsize=10)

# --- Panel C: Temporal Trend / Reference Year Distribution ---
if unique_pmid_years:
    # Premium histogram with a smooth KDE line for cited paper years
    sns.histplot(unique_pmid_years, bins=max(10, min(30, max_yr - min_yr)), kde=True, ax=axes[2], color='#2ca02c', edgecolor='black', alpha=0.6)
    
    # Highlight and label the median year with a distinct indicator line
    if pd.notna(med_yr):
        axes[2].axvline(med_yr, color='#d62728', linestyle='--', linewidth=2, label=f'Median: {int(med_yr)}')
        axes[2].legend(fontsize=11, loc='upper left')
    
    axes[2].set_xlabel('Publication Year', fontweight='bold', fontsize=12)
    axes[2].set_ylabel('Number of Unique PMIDs', fontweight='bold', fontsize=12)
    axes[2].set_title('Publication Year Distribution of Cited PMIDs', fontweight='bold', fontsize=13, pad=15)
    axes[2].tick_params(axis='both', which='major', labelsize=11)
else:
    axes[2].text(0.5, 0.5, "No Year Information Available", ha='center', va='center', fontsize=14, color='gray')
    axes[2].set_title('Publication Year Distribution', fontweight='bold', fontsize=13, pad=15)

plt.tight_layout()
plt.show()

# 6. Detailed Tier breakdown with Median Publication Year
print("\n=== Literature Evidence & Temporal Statistics by Tier ===")
tier_counts = df.groupby('Confidence_Tier')['has_evidence'].agg(['count', 'sum'])
for tier in ['Tier 1 (High)', 'Tier 2 (Medium)', 'Tier 3 (Low)']:
    if tier in tier_counts.index:
        cnt = tier_counts.loc[tier, 'count']
        ev = tier_counts.loc[tier, 'sum']
        pct = (ev / cnt) * 100 if cnt > 0 else 0
        
        # Calculate median publication year for references in this tier
        tier_df = df[df['Confidence_Tier'] == tier]
        tier_years = []
        for yrs in tier_df['evidence_years'].dropna():
            tier_years.extend(yrs)
            
        if tier_years:
            tier_med = np.nanmedian(tier_years)
            med_yr_str = f"Median Reference Year: {int(tier_med)}" if pd.notna(tier_med) else "No reference years"
        else:
            med_yr_str = "No reference years"
        print(f"  * {tier:18}: {ev:4,} / {cnt:5,} pairs ({pct:6.2f}%) | {med_yr_str}")

# 7. Show temporal landmark references (Oldest vs Newest in database)
if unique_pmid_years and os.path.exists(pubmed_results_path):
    print("\n=== Landmark References in Literature Evidence ===")
    
    # Sort PMIDs by year from metadata
    valid_meta = pubmed_df.dropna(subset=['Clean_Year']).copy()
    if not valid_meta.empty:
        # Find row with minimum year
        oldest_row = valid_meta.loc[valid_meta['Clean_Year'].idxmin()]
        print(f"  * Oldest Reference ({int(oldest_row['Clean_Year'])}): PMID {oldest_row['PMID']} - \"{oldest_row['Title']}\"")
        print(f"    Journal: {oldest_row['Journal']} | Authors: {oldest_row['Author']}\n")
        
        # Find row with maximum year
        newest_row = valid_meta.loc[valid_meta['Clean_Year'].idxmax()]
        print(f"  * Newest Reference ({int(newest_row['Clean_Year'])}): PMID {newest_row['PMID']} - \"{newest_row['Title']}\"")
        print(f"    Journal: {newest_row['Journal']} | Authors: {newest_row['Author']}")

---

## 5. Interaction Biology

### 5.1. Sensor Type Distribution

#### Purpose
To categorize the functional roles of target proteins (Receptors, Channels, Transporters, Enzymes) mediating metabolic communication. Metabolites act through distinct biological mechanisms: triggering intracellular signaling (Receptors), gating selective transport (Channels), internalizing cargo (Transporters), or undergoing chemical processing (Enzymes).

#### Database Origins & Multi-Database Mapping
To achieve high-fidelity and comprehensive functional profiling, we split and enriched target classifications across multiple database layers:
1. **`OtherDB_Sensor_Type`**: The original classifications integrated from the merged databases of **MEBOCOST** (metabolite-sensor maps), **CellPhoneDB v5** (ligand-receptor metadata), and **Celllinker2** (transporter-mediated pairings).
2. **`GtoPdb_Sensor_Type`**: Classifications retrieved from the IUPHAR/BPS **Guide to Pharmacology** mapping, mapping gene symbols to established pharmacological target classes.
3. **`HGNC_Sensor_Type`**: Gene group tags and standardized functional definitions retrieved from the **HGNC (HUGO Gene Nomenclature Committee)** Approved database.
4. **`UniProt_Sensor_Type`**: Classifications extracted dynamically from **UniProt KB** keywords and Gene Ontology (GO) terms using a high-throughput API batch query pipeline (enriched with 2,000+ cached gene symbols).
5. **`Sensor_Type` (Unified)**: The consolidated, deduplicated functional annotation that merges all the above sources and resolves multi-valued annotations row-wise (e.g., when a target acts as both a receptor and enzyme).

#### 🔮 Local Rule Classifier (classify_by_rules) Guide
To ensure **100% annotation completeness** for novel, rare, or unmapped targets, a regular-expression-based heuristic classifier serves as the final fallback layer in the pipeline. These rules recognize gene family prefixes and standard suffixes to assign correct functional roles:
- **Channels**: Matches prefixes such as `CACN*`, `SCN*`, `KCN*`, `CLCN*`, `TRP*`, `AQP*`, `ANO*`, `BEST*`, `PANX*`, `RYR*`, `ITPR*`, `ASIC*`, `GRIN*`, `GRIA*`, `GABR*`, `CHRN*`, `GLYR*`, `P2RX*`, `CX*`, `GJ*` or descriptors containing `"channel"` or `"pore"`.
- **Receptors**: Matches GPCR and nuclear receptor families including `GPR*`, `HRH*`, `ADR*`, `DRD*`, `HTR*`, `OPR*`, `P2RY*`, `LPAR*`, `S1PR*`, `FFAR*`, `CNR*`, `FZD*`, `SMO*`, `EGFR*`, `FGFR*`, `VEGFR*`, `NR*`, `THRA*`, `PPARA*`, `ESR*`, `AR*`, `GR*`, `MR*` or names containing `"receptor"`.
- **Transporters**: Matches solute carriers and active transporters such as `SLC*`, `ABC*`, `ATP1A*`, `ATP2A*`, `ATP4A*`, `ATP7A*`, `TFRC*`, `FABP*` or names containing `"transporter"`, `"carrier"`, or `"solute carrier"`.
- **Enzymes**: Matches metabolizing and signaling enzyme families including `CYP*`, `ALDH*`, `ADH*`, `COX*`, `PTGS*`, `NOS*`, `HSD*`, `SULT*`, `UGT*`, `GST*`, `MAO*`, `COMT*`, `PLA2*`, `PDE*`, `MAPK*`, `AKT*`, `JAK*`, `CDK*`, `CASP*`, `MMP*`, `DPP*` or suffixes ending in `"-ase"` (e.g. kinases, transferases, synthases, dehydrogenases).


In [ ]:
# 1. Print Coverage Comparison
print("=== Sensor Type Annotation Coverage ===")
cols_to_check = {
    "OtherDB_Sensor_Type": "Original DBs (OtherDB)",
    "GtoPdb_Sensor_Type": "Guide to Pharmacology (GtoPdb)",
    "HGNC_Sensor_Type": "HGNC approved gene groups",
    "UniProt_Sensor_Type": "UniProt KB annotations",
    "Sensor_Type": "Consolidated Unified (with regex fallbacks)"
}

coverage_data = {}
for col, label in cols_to_check.items():
    if col in df.columns:
        cnt = df[col].notna().sum()
        pct = (cnt / len(df)) * 100
        print(f"  * {label:42}: {cnt:5,} / {len(df):5,} pairs ({pct:6.2f}%)")
        coverage_data[label] = cnt
    else:
        print(f"  * {label:42}: Column '{col}' not found!")

# 2. Visualize Coverage Comparison
plt.figure(figsize=(12, 5))
cov_series = pd.Series(coverage_data).sort_values(ascending=True)
colors_cov = plt.cm.get_cmap('plasma')(np.linspace(0.2, 0.8, len(cov_series)))
bars = plt.barh(cov_series.index, cov_series.values, color=colors_cov, edgecolor='gray', height=0.6)
plt.title('Annotation Coverage Comparison by Database Source', fontweight='bold', fontsize=15, pad=15)
plt.xlabel('Number of Annotated Pairs', fontweight='bold', fontsize=12)
plt.xlim(0, len(df) * 1.15)
for bar in bars:
    width = bar.get_width()
    plt.text(width + 100, bar.get_y() + bar.get_height()/2, f"{width:,}\n({width/len(df)*100:.1f}%)", 
             va='center', ha='left', fontsize=10, fontweight='bold')
plt.tight_layout()
plt.show()

# 3. Explode and count multi-valued consolidated Sensor_Type values
sensor_elements = []
for val in df['Sensor_Type'].dropna():
    parts = [p.strip() for p in str(val).split(',') if p.strip()]
    sensor_elements.extend(parts)

st_counts = pd.Series(sensor_elements).value_counts()
print("\n=== Consolidated Unified Sensor Type Distribution (exploded multi-values) ===")
for cat, count in st_counts.items():
    print(f"  * {cat:12}: {count:5,} occurrences ({count/len(df)*100:6.2f}%)")

# 4. Plot Unified Sensor Type Distribution
plt.figure(figsize=(10, 5))
colors_dist = sns.color_palette('viridis', len(st_counts))
bars_dist = plt.bar(st_counts.index, st_counts.values, color=colors_dist, edgecolor='gray', width=0.4)
plt.title(f'Consolidated Unified Sensor Type Distribution (Exploded)', fontweight='bold', fontsize=15, pad=15)
plt.ylabel('Occurrence Count', fontweight='bold', fontsize=12)
plt.ylim(0, max(st_counts.values) * 1.15)
for bar in bars_dist:
    yval = bar.get_height()
    plt.text(bar.get_x() + bar.get_width()/2, yval + 100, f"{yval:,}\n({yval/len(df)*100:.1f}%)", 
             va='bottom', ha='center', fontsize=10, fontweight='bold')
plt.grid(axis='y', linestyle='--', alpha=0.7)
plt.tight_layout()
plt.show()


### 5.2. Enzyme Product/Substrate Relationships

#### Purpose
Focuses specifically on the subset of interactions classified as "Enzymes" to determine if the target protein treats the metabolite as a **substrate** (consuming it) or a **product** (synthesizing/releasing it). This defines the localized metabolic flux in the TME.

#### Database Origins & Multi-Database Mapping
Biochemical classifications are mapped across **four integrated layers** to determine how target enzymes interact with metabolic signals:
1. **`OtherDB_enzyme product/substrate`**: Original classifications from the enzyme tables of **CellLinker2** and **MRCLinkDB** and MEBOCOST mappings.
2. **`Reactions_enzyme product/substrate`**: Parsed from the `REACTIONS` column; metabolite names and synonyms are matched against reaction equation strings to infer substrate/product roles.
3. **`Rhea_enzyme product/substrate`**: Derived from [Rhea](https://www.rhea-db.org/), an expert-curated biochemical reaction database. UniProt -> Rhea reactions -> ChEBI compounds -> HMDB IDs pipeline (implemented in `annotate_enzyme_rhea.py`).
4. **`KEGG_enzyme product/substrate`**: Derived from [KEGG ENZYME](https://www.kegg.jp/kegg/rest/keggapi.html). UniProt -> KEGG gene -> EC number -> KEGG reaction -> KEGG compound -> ChEBI -> HMDB pipeline (implemented in `annotate_enzyme_kegg.py`).
5. **`enzyme product/substrate` (Unified)**: The consolidated, row-wise standardized and deduplicated biochemical classification across all four sources above. Values: `s` (substrate), `p` (product), or `s+p` (both).


In [ ]:
%matplotlib inline
import matplotlib.pyplot as plt

# 1. Print coverage for all enzyme annotation sources
print("=== Enzyme Product/Substrate Annotation Coverage ===")
cols_to_check_enz = {
    "OtherDB_enzyme product/substrate": "OtherDB (CellLinker2/MRCLinkDB)",
    "Reactions_enzyme product/substrate": "REACTIONS column parsing",
    "Rhea_enzyme product/substrate": "Rhea (UniProt->ChEBI->HMDB)",
    "KEGG_enzyme product/substrate": "KEGG ENZYME (UniProt->EC->HMDB)",
    "enzyme product/substrate": "Consolidated Unified"
}

coverage_data_enz = {}
for col, label in cols_to_check_enz.items():
    if col in df.columns:
        cnt = df[col].notna().sum()
        pct = (cnt / len(df)) * 100
        print(f"  * {label:35}: {cnt:5,} / {len(df):5,} pairs ({pct:6.2f}%)")
        coverage_data_enz[label] = cnt
    else:
        print(f"  * {label:35}: Column '{col}' not found in dataset")

# 2. Two-panel figure: coverage comparison + unified type distribution
if coverage_data_enz:
    fig, axes = plt.subplots(1, 2, figsize=(14, 5))

    # Left: coverage comparison
    labels_cov = list(coverage_data_enz.keys())
    values_cov = list(coverage_data_enz.values())
    colors_cov = sns.color_palette('Set2', len(labels_cov))
    bars_cov = axes[0].bar(range(len(labels_cov)), values_cov, color=colors_cov, edgecolor='gray', width=0.5)
    axes[0].set_xticks(range(len(labels_cov)))
    axes[0].set_xticklabels(labels_cov, rotation=30, ha='right', fontsize=9)
    axes[0].set_title('Enzyme Annotation Coverage by Source', fontweight='bold', fontsize=13, pad=12)
    axes[0].set_ylabel('Annotated Pairs', fontweight='bold', fontsize=11)
    axes[0].grid(axis='y', linestyle='--', alpha=0.7)
    for bar in bars_cov:
        yval = bar.get_height()
        axes[0].text(bar.get_x() + bar.get_width()/2, yval + 5, f"{yval:,}",
                     va='bottom', ha='center', fontsize=9, fontweight='bold')

    # Right: unified relationship types
    unified_col = 'enzyme product/substrate'
    if unified_col in df.columns:
        eps_clean = df[unified_col].value_counts(dropna=True)
        nan_count = df[unified_col].isna().sum()
        nan_pct = df[unified_col].isna().mean() * 100
        print(f"\nUnified Enzyme Relationships stats:")
        print(f"  * Unannotated pairs: {nan_count:,} ({nan_pct:.2f}%)")
        if len(eps_clean) > 0:
            colors_eps = sns.color_palette('Set2', len(eps_clean))
            bars_eps = axes[1].bar(eps_clean.index, eps_clean.values, color=colors_eps, edgecolor='gray', width=0.35)
            axes[1].set_title('Unified Enzyme-Metabolite Relationship Types', fontweight='bold', fontsize=13, pad=12)
            axes[1].set_ylabel('Occurrence Count', fontweight='bold', fontsize=11)
            axes[1].set_xlabel('Role (s=substrate, p=product, s+p=both)', fontweight='bold', fontsize=10)
            axes[1].set_ylim(0, max(eps_clean.values) * 1.15)
            axes[1].grid(axis='y', linestyle='--', alpha=0.7)
            for bar in bars_eps:
                yval = bar.get_height()
                axes[1].text(bar.get_x() + bar.get_width()/2, yval + 5, f"{yval:,}",
                             va='bottom', ha='center', fontsize=10, fontweight='bold')
    plt.tight_layout()
    plt.show()


### 5.3. Interaction Type (Promote/Inhibit/Release/Consume)

#### Purpose
Beyond simple binding, this section analyzes the functional outcome of the interaction in clinical oncology and cellular signaling contexts. It defines whether the metabolite-target pairing triggers activation (promote), downregulation (inhibit), secretion (release), or uptake (consume).

#### Database Origins & Multi-Database Mapping
Interaction dynamics are parsed and harmonized to analyze biological effects in the tumor microenvironment:
1. **`OtherDB_Interaction`**: Interaction directions originally retrieved from the clinical oncology tables of **MRCLinkDB** (`Metabolite-cell interaction.txt`), representing curated experimentally validated clinical and microenvironmental effects.
2. **`GtoPdb_Interaction`**: Interaction directions derived from the **IUPHAR/BPS Guide to Pharmacology** (`input/interactions.csv`), where target–ligand interaction Type and Action fields (e.g., Agonist, Antagonist, Inhibitor) are mapped to our standardized vocabulary using `map_gtopdb_action_to_interaction()` in `annotate_with_databases.py`. Matching is performed by PubChem SID or metabolite name against each target gene symbol's known ligand interactions.
3. **`Interaction` (Unified)**: The consolidated column produced by merging `OtherDB_Interaction` and `GtoPdb_Interaction` via `consolidate_interaction()`. Values are standardized to: `Promote`, `Inhibit`, `Be released`, `Be consumed`, `Regulate`, `Protect`.


In [ ]:
# 1. Print Coverage Comparison for Interaction annotations
print("=== Interaction Type Annotation Coverage ===")
cols_to_check_int = {
    "OtherDB_Interaction": "MRCLinkDB (OtherDB)",
    "GtoPdb_Interaction": "Guide to Pharmacology (GtoPdb)",
    "Interaction": "Consolidated Unified"
}

coverage_data_int = {}
for col, label in cols_to_check_int.items():
    if col in df.columns:
        cnt = df[col].notna().sum()
        pct = (cnt / len(df)) * 100
        print(f"  * {label:28}: {cnt:5,} / {len(df):5,} pairs ({pct:6.2f}%)")
        coverage_data_int[label] = cnt
    else:
        print(f"  * {label:28}: Column '{col}' not found!")

# 2. Plot Consolidated Interaction Type Distribution
inter = df['Interaction'].value_counts(dropna=False)
nan_count = df['Interaction'].isna().sum()
nan_percentage = df['Interaction'].isna().mean() * 100
print(f"\nUnified Interaction Directionality stats:")
print(f"  * Number of unannotated values: {nan_count:,} ({nan_percentage:.2f}%)")

inter_clean = df['Interaction'].value_counts(dropna=True)
colors_int = {
    'Promote': '#e74c3c', 
    'Inhibit': '#3498db', 
    'Be released': '#2ecc71', 
    'Be consumed': '#f39c12', 
    'Regulate': '#9b59b6', 
    'Protect': '#1abc9c'
}

plt.figure(figsize=(10, 5))
# Generate horizontal bar plot
bars_int = plt.barh(inter_clean.index, inter_clean.values, 
                    color=[colors_int.get(i, '#95a5a6') for i in inter_clean.index], 
                    edgecolor='gray', height=0.6)
plt.title(f'Unified Interaction Directionality Distribution', fontweight='bold', fontsize=14, pad=15)
plt.xlabel('Count', fontweight='bold', fontsize=11)
plt.ylabel('Interaction Type', fontweight='bold', fontsize=11)
plt.xlim(0, max(inter_clean.values) * 1.15)
for bar in bars_int:
    width = bar.get_width()
    plt.text(width + 5, bar.get_y() + bar.get_height()/2, f"{width:,}", 
             va='center', ha='left', fontsize=10, fontweight='bold')
plt.grid(axis='x', linestyle='--', alpha=0.7)
plt.tight_layout()
plt.show()


---

## 7. Cancer-Relevant Analysis

### 7.1. Disease Annotations

### Purpose
To map these metabolite-target interactions to clinically relevant diseases and cancer types using MRCLinkDB's expert-curated metadata. This grounds the basic biological network in real clinical oncology context.

**Data Provenance & Backtrack Info:**  
The metadata in columns `Disease`, `Cell type`, `Effect`, and `Interaction` is specific to the **MRCLinkDB** source. 

- **Source File:** `MRCLinkDB/Metabolite-cell interaction.txt`  
- **Ingestion Logic:** Handled in `scripts/merge_dbs_claude.py` within the `process_mrclinkdb()` function. The script specifically maps the clinical metadata provided by MRCLinkDB during the initial consolidation phase.  
- **Merge Strategy:** These annotations are carried through the pipeline only for pairs found in MRCLinkDB. For metabolites sourced from other databases (e.g., MEBOCOST, CellPhoneDBv5), these fields remain empty (`NaN`) as those databases do not provide clinical disease context in their raw ligand-receptor mapping.  

This backtrack ensures that any disease-specific results observed below can be directly attributed to the expert-curated interactions in the MRCLinkDB dataset.

### How to Interpret the Results
- **Disease Prevalence:** Shows which tumor types have the most highly characterized or literature-supported metabolite-target communication networks in public databases.
- **Annotated Pairs:** Allows you to filter down to interactions with known clinical relevance, focusing your downstream analysis on clinically validated targets.

In [ ]:
disease_vc = df['Disease'].value_counts(dropna=True)
plt.figure(figsize=(10, 10))
sns.barplot(x=disease_vc.values, y=disease_vc.index, palette='OrRd_r', hue=disease_vc.index, legend=False)
plt.title(f'Disease Annotations ({disease_vc.sum()} annotated pairs)'); plt.xlabel('Count')
plt.tight_layout(); plt.show()

cancer_terms = ['Cancer', 'cancer', 'carcinoma', 'Melanoma', 'melanoma', 'tumor', 'Tumor', 'leukemia', 'lymphoma']
cancer_mask = df['Disease'].str.contains('|'.join(cancer_terms), case=False, na=False)
print(f'\n→ {cancer_mask.sum()} pairs have explicit cancer/tumor disease annotations.')
print(f'→ Covering {df.loc[cancer_mask, "Metabolite_Name"].nunique()} metabolites and {df.loc[cancer_mask, "Target"].nunique()} targets.')

# ==========================================
# 💾 SAVE CANCER-ANNOTATED PAIRS WITH TIERS TO CSV
# ==========================================
df_cancer = df[cancer_mask].copy()
cancer_out_path = '../output/human_metab_target_pairs_cancer_annotated.csv'
df_cancer.to_csv(cancer_out_path, index=False)

print(f"\n🎉 Successfully saved {len(df_cancer)} cancer-annotated interaction pairs to:")
print(f"   -> '{cancer_out_path}'")

# 3. Display preview showing explicit mapping of metabolite and pair confidence tiers
df_preview = df_cancer[[
    'Metabolite_Name', 'Target', 'Disease', 'Cell type', 
    'Metabolite_Confidence_Tier', 'Pair_Confidence_Tier', 'databases_count'
]].head(15)
df_preview

### 7.2. Cell Type Annotations

### Purpose
Investigates which cell types (e.g., immune cells, cancer cells, stromal cells) are annotated as active senders or receivers for these interaction pairs. This helps reconstruct the multi-cellular cellular network of the tumor microenvironment (TME).

### How to Interpret the Results
- **Stromal-Tumor Axis:** Identifies how fibroblasts, endothelial cells, or extracellular matrix components talk to cancer cells.
- **Immune-Tumor Axis:** Pinpoints how immune cells sense metabolic shifts (e.g., T-cell suppression via lactate/adenosine) in the TME.

In [ ]:
ct = df['Cell type'].value_counts(dropna=True)
plt.figure(figsize=(10, 8))
sns.barplot(x=ct.values, y=ct.index, palette='Greens_r', hue=ct.index, legend=False)
plt.title(f'Cell Type Context ({ct.sum()} annotated pairs)'); plt.xlabel('Count')
plt.tight_layout(); plt.show()

### 7.3. Cancer Cell Interactions: Effect & Directionality

### Purpose
To specifically isolate interactions with explicit cancer cell annotations to profile their functional outcome (e.g., promoting proliferation, inducing drug resistance, or triggering apoptosis) and directional flow.

### How to Interpret the Results
- **Pro-Tumorigenic Effects (e.g., "Promotes proliferation/metastasis"):** Ranks the key target pairs driving tumor progression, indicating high-priority therapeutic targets.
- **Immunosuppressive Directionality:** Highlights metabolite fluxes that silence cytotoxic immune cells, representing ideal targets for combination immunotherapy.

In [ ]:
cancer_df = df[cancer_mask].copy()
if len(cancer_df) > 0:
    fig, axes = plt.subplots(1, 2, figsize=(14, 5))
    # Effect types in cancer
    eff = cancer_df['Effect'].value_counts(dropna=True)
    sns.barplot(x=eff.values, y=eff.index, palette='RdYlBu_r', hue=eff.index, legend=False, ax=axes[0])
    axes[0].set_title('Effect Types in Cancer Pairs')
    # Interaction direction in cancer
    inter_c = cancer_df['Interaction'].value_counts(dropna=True)
    if len(inter_c) > 0:
        sns.barplot(x=inter_c.values, y=inter_c.index, palette='coolwarm', hue=inter_c.index, legend=False, ax=axes[1])
    axes[1].set_title('Interaction Direction in Cancer Pairs')
    plt.tight_layout(); plt.show()

    print('\n=== Top Cancer-Associated Metabolites ===')
    print(cancer_df['Metabolite_Name'].value_counts().head(10).to_string())
    print('\n=== Top Cancer-Associated Targets ===')
    print(cancer_df['Target'].value_counts().head(10).to_string())
else:
    print('No cancer-annotated pairs found.')

---

## 8. CCC Readiness: Sender–Receiver Gene Pairs

### Purpose
For cell-cell communication scoring tools (like CellChat, SingleCellSignalR, or MEBOCOST), an interaction pair is only fully usable if we can measure both the sender cell's capacity to synthesize the metabolite (`Synthetic_genes`) and the receiver cell's capacity to sense it (`Sensor_Gene` or `Target`). This section filters and validates pairs that are fully equipped with both gene annotations.

### How to Interpret the Results
- **Dual-Gene Annotated Pairs:** These represent the most robust, computationally-ready interaction channels. They are directly ready to be merged with single-cell RNA-sequencing (scRNAseq) datasets to calculate spatial or cellular communication scores.

In [ ]:
has_sender = df['Synthetic_genes'].notna()
has_receiver = df['Sensor_Gene'].notna() | df['Target'].notna()
ccc_ready = has_sender & has_receiver

print(f'Total pairs: {len(df):,}')
print(f'Has sender gene (Synthetic_genes): {has_sender.sum():,} ({has_sender.mean()*100:.1f}%)')
print(f'Has receiver gene (Sensor_Gene):    {df["Sensor_Gene"].notna().sum():,}')
print(f'CCC-ready pairs (sender + receiver): {ccc_ready.sum():,} ({ccc_ready.mean()*100:.1f}%)')

# Venn-like summary
fig, ax = plt.subplots(figsize=(8, 5))
cats = ['Sender only', 'Receiver only', 'Both (CCC-ready)', 'Neither']
vals = [
    (has_sender & ~df['Sensor_Gene'].notna()).sum(),
    (~has_sender & df['Sensor_Gene'].notna()).sum(),
    (has_sender & df['Sensor_Gene'].notna()).sum(),
    (~has_sender & ~df['Sensor_Gene'].notna()).sum()
]
colors_v = ['#3498db', '#e67e22', '#2ecc71', '#bdc3c7']
ax.bar(cats, vals, color=colors_v)
for i, v in enumerate(vals): ax.text(i, v, f'{v:,}', ha='center', va='bottom')
ax.set_ylabel('# Pairs'); ax.set_title('CCC Readiness: Sender/Receiver Gene Availability')
plt.tight_layout(); plt.show()

---

## 9. High-Confidence Multi-DB Pairs

### Purpose
Integrates the filters (consensus across 3+ databases AND complete gene annotations) to isolate a highly curated, ultra-high-confidence catalog of metabolic communication links.

### How to Interpret the Results
- **High-Confidence Subset:** These are your prime biological targets. They represent the "gold standard" interactions where the databases fully agree on the molecules involved, and the genetic architecture is completely characterized.

In [ ]:
hc = df[(df['databases_count'] >= 3) & ccc_ready].copy()
print(f'High-confidence CCC-ready pairs: {len(hc):,}')
print(f'  Unique metabolites: {hc["Metabolite_Name"].nunique()}')
print(f'  Unique targets: {hc["Target"].nunique()}')

if len(hc) > 0:
    print('\n=== Top Metabolites in High-Confidence Set ===')
    print(hc['Metabolite_Name'].value_counts().head(15).to_string())
    print('\n=== Top Targets in High-Confidence Set ===')
    print(hc['Target'].value_counts().head(15).to_string())

---

## 10. Target Gene Lists for Downstream Integration

### Purpose
To extract the clean, unique lists of target genes (Receptors, Solute Carriers, Enzymes) that mediate these high-confidence interactions. These gene lists are ready to be integrated with clinical databases (like TCGA or GEO) to run survival analyses or expression lookups.

### How to Interpret the Results
- **Gene Lists:** These lists represent your target library. High expression of receptor genes in patient samples can be correlated with survival outcomes or drug responsiveness.

In [ ]:
# All unique target genes
all_targets = df['Target'].dropna().unique()
print(f'All unique target genes: {len(all_targets)}')

# Sender genes
sender_genes = df['Synthetic_genes'].dropna().str.split(',').explode().str.strip().unique()
print(f'All unique sender genes: {len(sender_genes)}')

# Sensor genes
sensor_genes = df['Sensor_Gene'].dropna().unique()
print(f'All unique sensor genes: {len(sensor_genes)}')

# Save for downstream
os.makedirs('../output/gene_lists', exist_ok=True)
pd.Series(all_targets).to_csv('../output/gene_lists/all_target_genes.txt', index=False, header=False)
pd.Series(sender_genes).to_csv('../output/gene_lists/sender_genes.txt', index=False, header=False)
pd.Series(sensor_genes).to_csv('../output/gene_lists/sensor_genes.txt', index=False, header=False)
print('\n→ Gene lists saved to output/gene_lists/ for TCGA/GEO queries.')

---

## 11. Summary Statistics

### Purpose
Provides a high-level statistical scorecard summarizing the dataset size, average degree, and confidence tier breakdown.

### How to Interpret the Results
- **Consensus Metrics:** Serves as a reference card for scientific publications or presentations, summarizing the structural properties of your consolidated network.

In [ ]:
summary = {
    'Total interaction pairs': f'{len(df):,}',
    'Unique HMDB metabolites': df['HMDB_ID'].nunique(),
    'Unique metabolite names': df['Metabolite_Name'].nunique(),
    'Unique targets': df['Target'].nunique(),
    'Cancer-annotated pairs': cancer_mask.sum(),
    'CCC-ready pairs (sender+receiver)': ccc_ready.sum(),
    'High-conf CCC-ready (3+ DB)': len(hc),
    'With Sensor_Type annotation': df['Sensor_Type'].notna().sum(),
    'With Disease annotation': df['Disease'].notna().sum(),
    'With scCellFie score': df['scCellFie_value'].notna().sum(),
    'With Evidence_Score': df['Evidence_Score'].notna().sum(),
    'Pairs with Literature Evidence (PMID)': df['has_evidence'].sum() if 'has_evidence' in df.columns else 0,
    'Total unique PMIDs cited': len(set(p.strip() for p in df['Cleaned_PMID'].dropna().str.split(';').explode().unique() if p.strip())) if 'Cleaned_PMID' in df.columns else 0,
    'Median Publication Year of Citations': int(np.nanmedian([pmid_to_year[p] for p in set(p.strip() for p in df['Cleaned_PMID'].dropna().str.split(';').explode().unique() if p.strip()) if 'pmid_to_year' in globals() and p in pmid_to_year and pd.notna(pmid_to_year[p])])) if ('Cleaned_PMID' in df.columns and 'pmid_to_year' in globals() and any(p in pmid_to_year and pd.notna(pmid_to_year[p]) for p in set(p.strip() for p in df['Cleaned_PMID'].dropna().str.split(';').explode().unique() if p.strip()))) else 'N/A',
}
print(pd.DataFrame.from_dict(summary, orient='index', columns=['Value']))


---

## 12. Export Full Notebook Report to HTML

Compiling this entire interactive notebook—including all structured explanations, executable code blocks, data tables, and high-resolution plots—into a single publication-ready and highly interactive HTML report.

In [ ]:
# ==========================================
# 📄 FULL NOTEBOOK HTML REPORT EXPORT
# ==========================================
if 'SAVE_AS_HTML' in globals() and SAVE_AS_HTML:
    import subprocess
    import sys
    import os

    notebook_filename = "metab_targetPair_analysis.ipynb"

    output_dir = "../output"
    os.makedirs(output_dir, exist_ok=True)

    output_name = f"metab_targetPair_analysis_full_report_{TIER_FILTER}"

    print(f"Exporting notebook '{notebook_filename}' to HTML...")

    jupyter_bin = os.path.join(
        os.path.dirname(sys.executable),
        "jupyter"
    )

    if not os.path.exists(jupyter_bin):
        jupyter_bin = "jupyter"

    cmd_html = [
        jupyter_bin,
        "nbconvert",
        "--to",
        "html",
        notebook_filename,
        "--output",
        output_name,
        "--output-dir",
        output_dir,
    ]

    print("Running:")
    print(" ".join(cmd_html))

    res_html = subprocess.run(
        cmd_html,
        capture_output=True,
        text=True
    )

    if res_html.returncode == 0:
        output_file = os.path.join(
            output_dir,
            output_name + ".html"
        )

        print("🎉 SUCCESS!")
        print(f"Saved HTML report to:\n{output_file}")

    else:
        print("❌ HTML export failed")
        print("\nSTDOUT:")
        print(res_html.stdout)
        print("\nSTDERR:")
        print(res_html.stderr)

else:
    print(
        "Full notebook HTML export is disabled. "
        "Set SAVE_AS_HTML = True to enable."
    )